[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranskills/genai-playground/blob/main/notebooks/00-models-exploration.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/notebooks/welcome?src=https://github.com/ranskills/genai-playground/blob/main/notebooks/00-models-exploration.ipynb)
[![Open In Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/ranskills/genai-playground/blob/main/notebooks/00-models-exploration.ipynb)

# Free-tier Model Access with OpenAI API compatibility

In [ ]:
!pip install -qU openai ipywidgets

In [ ]:
import logging
import os
import sys
import random
from enum import StrEnum
from collections import defaultdict

from openai import OpenAI
from dotenv import load_dotenv


load_dotenv(override=True)

### Log Setup

In [ ]:
logging.basicConfig(level=logging.WARNING)

valid_levels = ['DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL']
log_level = os.environ.get('LOG_LEVEL', 'INFO').upper()
log_level = log_level.upper() if log_level in valid_levels else 'DEBUG'

logger = logging.getLogger('simu-learn')
logger.setLevel(log_level)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        fmt='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

## Utilities

### Secrets

In [ ]:
def _get_secret_from_environment(name: str) -> str:
  secret = os.environ.get(name, '').strip()

  if secret:
      print(f'✅ Environment Variable: Found {name}')
      return secret

  print(f'❌ Environment Variable : {name} is not set')

  try:
    from google.colab import userdata

    secret = userdata.get(name)
    print(f'✅ Google Colab: Found {name}')
  except Exception as e:
    print(f'❌ Google Colab: {e}')

  return secret.strip()


def _get_secret_from_user(name: str) -> str:
  from getpass import getpass

  return getpass(f'Enter the secret value for {name}: ')

def get_secret(name: str) -> str:
  secret = _get_secret_from_environment(name)
  if not secret:
    secret = _get_secret_from_user(name)

  return secret

## Setup LLM Providers



### Free-tier API Keys 🔑

These generous providers give access to API Keys on a free-tier plan

- https://ollama.com/
- https://www.cerebras.ai/
- https://openrouter.ai/
- https://console.groq.com/keys
- https://aistudio.google.com
- https://huggingface.co/ - Create an `Access Token`

### Code

In [ ]:
class Provider(StrEnum):
    OLLAMA = 'Ollama'
    OLLAMA_LOCAL = 'Ollama (Local)'
    OPENAI = 'OpenAI'
    OPENROUTER = 'OpenRouter'
    CEREBRAS = 'Cerebras'
    HUGGINGFACE = 'HuggingFace'
    GOOGLE='Google'
    GROQ='Groq'


available_providers = [item.value for item in Provider]
provider_config: dict[Provider, tuple[str, str]] = {
    Provider.OLLAMA: ('OLLAMA_API_KEY', 'https://ollama.com/v1'),
    Provider.OLLAMA_LOCAL: ('OLLAMA_API_KEY', os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')),
    Provider.OPENAI: ('OPENAI_API_KEY', 'https://api.openai.com/v1'),
    Provider.OPENROUTER: ('OPENROUTER_API_KEY', 'https://openrouter.ai/api/v1'),
    Provider.CEREBRAS: ('CEREBRAS_API_KEY', 'https://api.cerebras.ai/v1'),
    Provider.HUGGINGFACE: ('HF_TOKEN', 'https://router.huggingface.co/v1'),
    Provider.GOOGLE: ('GOOGLE_API_KEY', 'https://generativelanguage.googleapis.com/v1beta/openai/'),
    Provider.GROQ: ('GROQ_API_KEY', 'https://api.groq.com/openai/v1'),
}

clients: dict[Provider, OpenAI] = {}

models: dict[Provider, list[str]] = defaultdict(list)

selection_state: dict[Provider, str | None] = {
    Provider.OLLAMA: 'gpt-oss:20b',
    Provider.OLLAMA_LOCAL: 'llama3.2:3b',
    Provider.OPENAI: 'gpt-4o-mini',
    Provider.OPENROUTER: 'openai/gpt-oss-20b:free',
    Provider.CEREBRAS: 'gpt-oss-120b',
    Provider.HUGGINGFACE: None,
    Provider.GOOGLE: None,
    Provider.GROQ: None,
}

DEFAULT_PROVIDER = Provider.CEREBRAS

selected_provider, selected_model, client = '', '', None


def get_desired_value_or_first_item(desire, options) -> str | None:
    logger.debug(f'Pick {desire} from {options}')
    selected = desire if desire in options else None
    if selected:
        return selected

    return options[0] if options else None


try:
    selected_provider = get_desired_value_or_first_item(DEFAULT_PROVIDER, available_providers)
    selected_provider = Provider(selected_provider)
    client = clients.get(selected_provider)
except Exception:
    logger.warning(f'❌ no provider configured and everything else from here will FAIL 🤦, I know you know this already.')


def setup_provider_client(provider: Provider) -> OpenAI | None:
    global provider_config, clients

    client = clients.get(provider, None)
    logger.debug(f'Client status for {provider}: has client: {True if client else False}')

    if client:
        return client

    key_name, base_url = provider_config.get(provider)


    api_key = 'ollama' if provider == Provider.OLLAMA_LOCAL else get_secret(key_name)

    if not api_key:
        return None

    client = OpenAI(api_key=api_key, base_url=base_url)
    clients[provider] = client

    return client


def load_models_if_needed(selected_provider: Provider):
    global selected_model, models, provider_config, clients, client

    provider = selected_provider
    client = setup_provider_client(provider)

    if client and not models.get(selected_provider):
        logging.info(f'📡 Fetching {selected_provider} models...')

        models[selected_provider] = [model.id for model in client.models.list()]
        selected_model = get_desired_value_or_first_item(
            selection_state[selected_provider],
            models[selected_provider],
        )


load_models_if_needed(selected_provider)
client = clients.get(selected_provider, None)

logger.info(f'ℹ️ Provider: {selected_provider} Model: {selected_model}, Client: {client}')

In [ ]:
get_desired_value_or_first_item(selection_state[selected_provider], models[selected_provider])
selection_state[selected_provider]
models

### Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display

output_area = widgets.Output(
    layout={
        'width': '100%',
        'height': '300px',
        'overflow_y': 'auto',
        'border': '1px solid #ddd',
        'padding': '10px',
        'background': '#f9f9f9'
    }
)

# print('XXX', selected_model)
provider_selector = widgets.Dropdown(
    options=available_providers,
    value=get_desired_value_or_first_item(selected_provider, available_providers),
)

model_selector = widgets.Dropdown(
    options=models.get(selected_provider, []),
    value=get_desired_value_or_first_item(selection_state[selected_provider], models[selected_provider]),
)


def provider_change_handler(change):
    global selected_provider, client, models

    with output_area:

        logger.info(f'Provider changed to {change.new}')
        print(f'Provider changed to {change.new}')
        selected_provider = Provider(change.new)
        load_models_if_needed(selected_provider)

        xxx = selection_state[selected_provider]
        print(f'>>> {xxx}')
        model_selector.options = models.get(selected_provider, [])
        yyy = get_desired_value_or_first_item(xxx, models[selected_provider])
        model_selector.value = yyy
        selection_state[selected_provider] = yyy
        # model_selector.value = get_desired_value_or_first_item(selection_state[selected_provider], models[selected_provider])


def model_change_handler(change):
    global selected_provider, selected_model, selection_state

    with output_area:
        selected_model = change.new
        selection_state[selected_provider] = selected_model
        logger.info(f'👉 Selected model: {selected_model} Provider: {selected_provider}')


provider_selector.observe(provider_change_handler, names='value')
model_selector.observe(model_change_handler, names='value')

model_component = widgets.HBox(children=[provider_selector, model_selector])

display(model_component)
display(output_area)